# Therapy AI Fine-Tuning - Clean Version

This notebook fine-tunes open language models for university student mental health support.

## ⚠️ IMPORTANT: Run cells in order!

1. **Cell 1**: Install packages
2. **Cell 2**: Import libraries  
3. **Cell 3**: Load dataset
4. **Cell 4**: Configure model
5. **Cell 5**: Configure LoRA
6. **Cell 6**: Configure training
7. **Cell 7**: Start training
8. **Cell 8**: Test model

## 💻 Hardware Requirements:
- **CPU**: Any modern CPU (training will be slower)
- **GPU**: Any CUDA-compatible GPU (recommended for faster training)
- **RAM**: 8GB+ recommended
- **Storage**: 5GB+ free space


In [ ]:
# Install required packages
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
%pip install -q transformers datasets accelerate peft bitsandbytes trl
%pip install -q huggingface_hub

print("✅ All packages installed successfully!")


^C
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
✅ All packages installed successfully!



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


ERROR: Exception:
Traceback (most recent call last):
  File "c:\Users\HP USERS\AppData\Local\Programs\Python\Python311\Lib\site-packages\pip\_vendor\urllib3\response.py", line 438, in _error_catcher
    yield
  File "c:\Users\HP USERS\AppData\Local\Programs\Python\Python311\Lib\site-packages\pip\_vendor\urllib3\response.py", line 561, in read
    data = self._fp_read(amt) if not fp_closed else b""
           ^^^^^^^^^^^^^^^^^^
  File "c:\Users\HP USERS\AppData\Local\Programs\Python\Python311\Lib\site-packages\pip\_vendor\urllib3\response.py", line 527, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
           ^^^^^^^^^^^^^^^^^^
  File "c:\Users\HP USERS\AppData\Local\Programs\Python\Python311\Lib\site-packages\pip\_vendor\cachecontrol\filewrapper.py", line 98, in read
    data: bytes = self.__fp.read(amt)
                  ^^^^^^^^^^^^^^^^^^^
  File "c:\Users\HP USERS\AppData\Local\Programs\Python\Python311\Lib\http\client.py", line 473, in read
    s

In [3]:
# Import all required libraries
import torch
import json
import os
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer
import warnings
warnings.filterwarnings("ignore")

# Check GPU availability and CUDA setup
print(f"🚀 CUDA available: {torch.cuda.is_available()}")
print(f"🐍 Python version: {torch.version.python}")
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"⚡ CUDA version (PyTorch): {torch.version.cuda}")

if torch.cuda.is_available():
    print(f"📱 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"🔢 GPU Count: {torch.cuda.device_count()}")
else:
    print("⚠️ No GPU detected - training will be slower on CPU")
    print("🔧 Troubleshooting:")
    print("   1. Check if NVIDIA drivers are installed")
    print("   2. Check if CUDA toolkit is installed")
    print("   3. Try: pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118")

print("✅ All imports successful!")


🚀 CUDA available: False


AttributeError: module 'torch.version' has no attribute 'python'

In [4]:
# Load and prepare dataset
print("📊 Loading dataset...")

# Check if running on Kaggle or locally
if os.path.exists('/kaggle/input/therapy-dataset/therapy_fine_tuning_dataset.jsonl'):
    dataset_path = '/kaggle/input/therapy-dataset/therapy_fine_tuning_dataset.jsonl'
    print("🔍 Running on Kaggle")
else:
    dataset_path = 'therapy_fine_tuning_dataset.jsonl'
    print("🔍 Running locally")
    
    if not os.path.exists(dataset_path):
        print(f"❌ Dataset file not found: {dataset_path}")
        print("📁 Please make sure 'therapy_fine_tuning_dataset.jsonl' is in the same directory")
        raise FileNotFoundError(f"Dataset file not found: {dataset_path}")

dataset = load_dataset('json', data_files=dataset_path, split='train')
print(f"📊 Loaded dataset: {len(dataset)} conversations")

# Format conversations for training (TinyLlama chat format)
def format_conversation(example):
    messages = example['messages']
    text = ""
    for message in messages:
        if message['role'] == 'user':
            text += f"<|user|>\n{message['content']}\n"
        elif message['role'] == 'model':
            text += f"<|assistant|>\n{message['content']}\n"
    return {"text": text}

dataset = dataset.map(format_conversation)
print("✅ Dataset formatted for training!")
print(f"📝 Sample: {dataset[0]['text'][:100]}...")


📊 Loading dataset...
🔍 Running locally
📊 Loaded dataset: 110 conversations
✅ Dataset formatted for training!
📝 Sample: <|user|>
Hello, I'm feeling really stressed about my studies
<|assistant|>
Hello! I'm here to listen...


In [ ]:
# Configure model and tokenizer
print("🔄 Loading model and tokenizer...")

# Using TinyLlama - perfect for fine-tuning!
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
new_model_name = "therapy-ai-tinyllama-clean"

print(f"🎯 Selected model: {model_name}")
print(f"💾 Output name: {new_model_name}")
print("🚀 TinyLlama is perfect for fine-tuning - small, fast, and open!")

# Skip quantization for standard fine-tuning
print("🔄 Loading model without quantization...")

# Load model without device_map to avoid accelerate conflicts
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)
model.config.use_cache = False

# Move model to device manually
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"✅ Model loaded and moved to: {device}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("✅ Model and tokenizer loaded successfully!")
print(f"🧠 Model: {model_name}")
print(f"📝 Tokenizer vocab size: {len(tokenizer)}")


🔄 Loading model and tokenizer...
🎯 Selected model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
💾 Output name: therapy-ai-tinyllama-clean
🚀 TinyLlama is perfect for fine-tuning - small, fast, and open!
🔄 Loading model without quantization...


`torch_dtype` is deprecated! Use `dtype` instead!


✅ Model loaded and moved to: cpu
✅ Model and tokenizer loaded successfully!
🧠 Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
📝 Tokenizer vocab size: 32000


In [7]:
# Configure LoRA
print("🎯 Configuring LoRA...")

# Get the actual model architecture to determine correct target modules
print(f"🔍 Model type: {type(model).__name__}")
print(f"🔍 Model config: {model.config.model_type}")

# Define target modules based on model type
if "dialogpt" in model_name.lower():
    # DialoGPT uses different module names
    target_modules = ["c_attn", "c_proj", "c_fc"]
    print("🎯 Using DialoGPT target modules")
elif "gpt2" in model_name.lower():
    # GPT-2 uses similar modules to DialoGPT
    target_modules = ["c_attn", "c_proj", "c_fc"]
    print("🎯 Using GPT-2 target modules")
elif "blenderbot" in model_name.lower():
    # BlenderBot uses different modules
    target_modules = ["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"]
    print("🎯 Using BlenderBot target modules")
else:
    # Default to common transformer modules
    target_modules = ["q_proj", "v_proj", "k_proj", "o_proj"]
    print("🎯 Using default target modules")

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules
)

print("✅ LoRA configuration ready!")
print(f"🎯 Target modules: {target_modules}")
print(f"📊 LoRA rank: {peft_config.r}")


🎯 Configuring LoRA...
🔍 Model type: LlamaForCausalLM
🔍 Model config: llama
🎯 Using default target modules
✅ LoRA configuration ready!
🎯 Target modules: ['q_proj', 'v_proj', 'k_proj', 'o_proj']
📊 LoRA rank: 64


In [9]:
# Configure training arguments
print("⚙️ Configuring training...")

# Optimize for Quadro P1000 (4GB VRAM)
use_gpu = torch.cuda.is_available()
print(f"🖥️ GPU Available: {use_gpu}")
if use_gpu:
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

training_args = TrainingArguments(
    output_dir="./therapy-ai-results",
    num_train_epochs=3,
    per_device_train_batch_size=1,  # Small batch for 4GB VRAM
    gradient_accumulation_steps=4,  # Effective batch size = 4
    optim="adamw_torch",
    logging_steps=10,
    learning_rate=2e-4,
    fp16=use_gpu,  # Enable fp16 on GPU for memory savings
    bf16=False,    # P1000 doesn't support bf16
    save_steps=100,
    save_total_limit=2,
    remove_unused_columns=False,
    warmup_steps=100,
    max_grad_norm=0.3,
    lr_scheduler_type="cosine",
    report_to="none",
    dataloader_pin_memory=use_gpu,  # Enable pin memory on GPU
    dataloader_num_workers=0,       # Reduce memory usage
)

print("✅ Training arguments configured!")
print(f"🎯 Epochs: {training_args.num_train_epochs}")
print(f"📊 Batch size: {training_args.per_device_train_batch_size}")
print(f"📈 Learning rate: {training_args.learning_rate}")


⚙️ Configuring training...
🖥️ GPU Available: False
✅ Training arguments configured!
🎯 Epochs: 3
📊 Batch size: 1
📈 Learning rate: 0.0002


In [10]:
# Initialize trainer and start training
print("🚀 Initializing trainer...")

# Create a proper data collator for the quantized model
def data_collator(batch):
    # Tokenize the text data
    texts = [item['text'] for item in batch]
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )
    inputs['labels'] = inputs['input_ids'].clone()
    return inputs

# Skip LoRA for now and use standard fine-tuning
print("🔄 Skipping LoRA due to compatibility issues...")
print("🔄 Using standard fine-tuning approach...")

# Use standard Trainer without PEFT
from transformers import Trainer
trainer = Trainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    data_collator=data_collator,
)
print("✅ Using standard Trainer (no LoRA)")

print("✅ Trainer initialized successfully!")
print(f"📊 Training samples: {len(dataset)}")

print("\n🏃 Starting fine-tuning...")
print("⏱️ This may take several hours depending on your hardware")

# Start training
trainer.train()

print("✅ Training completed successfully!")

# Save model
trainer.save_model(new_model_name)
tokenizer.save_pretrained(new_model_name)
print(f"✅ Model saved as: {new_model_name}")


🚀 Initializing trainer...
🔄 Skipping LoRA due to compatibility issues...
🔄 Using standard fine-tuning approach...
✅ Using standard Trainer (no LoRA)
✅ Trainer initialized successfully!
📊 Training samples: 110

🏃 Starting fine-tuning...
⏱️ This may take several hours depending on your hardware


Step,Training Loss
10,1.356300
20,1.055000
30,0.739000
40,0.553400
50,0.619300
60,0.579200
70,0.722800
80,0.601500


✅ Training completed successfully!
✅ Model saved as: therapy-ai-tinyllama-clean


In [11]:
# Test the fine-tuned model
print("🧪 Testing the fine-tuned model...")

def test_model(prompt, max_length=200):
    # Format for TinyLlama
    formatted_prompt = f"<|user|>\n{prompt}\n<|assistant|>\n"
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the assistant response
    ai_response = response.split("<|assistant|>")[-1].strip()
    return ai_response

# Test with sample prompts
test_prompts = [
    "I'm feeling really stressed about my studies",
    "I'm having a panic attack right now",
    "I feel depressed and hopeless"
]

print("=" * 50)
for prompt in test_prompts:
    print(f"\n👤 User: {prompt}")
    response = test_model(prompt)
    print(f"🤖 AI: {response}")
    print("-" * 30)

print("\n🎉 Fine-tuning and testing completed successfully!")


🧪 Testing the fine-tuned model...

👤 User: I'm feeling really stressed about my studies
🤖 AI: I understand how stressful university can be, especially when you're feeling overwhelmed and behind in your studies. What specific aspects of university are causing you the most stress? Are you dealing with financial stress, emotional stress, or overwhelming others' expectations? It's completely understandable that university would be causing you the most stress. What strategies have you tried to address these feelings? Sometimes talking to mentors, students, and other stakeholders, or what resources have you explored? What would help you feel more supported during your studies? It's important to remember that you're not alone in feeling this way and that there are many resources available to you. What would help you feel more comfortable and safe in your university environment?
I'd strongly recommend reaching out to your university's student services and trusted friends, family members, or ot